# Sync parsed manuals into Vector Search

This notebook is the **indexing** step of the pipeline. It assumes the parsing notebook
(`02-parse pdf for ai search`) has already run and produced the Delta table
`{catalog}.{schema}.manual_document_with_images_parsed` (with Change Data Feed enabled).

Here we:
1. Create a Vector Search endpoint (if needed)
2. Create / sync a delta-sync index on top of the parsed table

In [0]:
%pip install -U -qqqq mlflow>=3.1.4 langchain==0.3.27 langgraph==0.6.11 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks] databricks-feature-engineering==0.12.1 protobuf<5  cryptography<43 databricks-mcp
dbutils.library.restartPython()

In [0]:
%run ./01-setup

In [0]:
# Configuration - catalog & schema are passed as job/DAB parameters (with defaults for interactive runs).
# Widgets survive the restartPython above, so it is safe to read them here.
dbutils.widgets.text("catalog", "stable_classic_6kvrb7_catalog", "Catalog")
dbutils.widgets.text("schema", "cesar_cordoba", "Schema")
dbutils.widgets.text("volume", "manuales", "Volume")

# The Vector Search endpoint is created by the DAB
dbutils.widgets.text("vector_search_endpoint", "technical_manuals_vs_endpoint", "Vector Search endpoint")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

VECTOR_SEARCH_ENDPOINT_NAME = dbutils.widgets.get("vector_search_endpoint")

print(f"Using catalog={catalog}, schema={schema}, endpoint={VECTOR_SEARCH_ENDPOINT_NAME}")

## 1/ Vector Search endpoint

The endpoint is the entry point that serves your indexes. Once created you can view it in the
[Vector Search Endpoints UI](#/setting/clusters/vector-search).

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

if not endpoint_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME):
    vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT_NAME, endpoint_type="STANDARD")

wait_for_vs_endpoint_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME)
print(f"Endpoint named {VECTOR_SEARCH_ENDPOINT_NAME} is ready.")

## 2/ Create (or sync) the Vector Search index

We build a **managed-embeddings** delta-sync index on the table produced by the parsing notebook.
Databricks computes the embeddings from the `chunk_to_embed` text column and keeps the index in
sync with the Delta table.

In [0]:
#The table we'd like to index (produced by 02-parse pdf for ai search)
source_table_fullname = f"{catalog}.{schema}.{volume}_document_with_images_parsed"
# Where we want to store our index
vs_index_fullname = f"{catalog}.{schema}.{volume}_knowledge_base_vs_index"

if not index_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname):
  print(f"Creating index {vs_index_fullname} on endpoint {VECTOR_SEARCH_ENDPOINT_NAME}...")
  vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=vs_index_fullname,
    source_table_name=source_table_fullname,
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_source_column='chunk_to_embed', #The column containing our text
    embedding_model_endpoint_name='databricks-gte-large-en' #The embedding endpoint used to create the embeddings
  )
  #Let's wait for the index to be ready and all our embeddings to be created and indexed
  wait_for_index_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)
else:
  #Trigger a sync to update our vs content with the new data saved in the table
  wait_for_index_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)
  vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname).sync()

print(f"index {vs_index_fullname} on table {source_table_fullname} is ready")

## 3/ Try the index: search for relevant content

Databricks automatically captures and synchronizes new entries in the table with the index.

In [0]:
question = "How do I connect the device to Wi-Fi?"

results = vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname).similarity_search(
  query_text=question,
  columns=["id", "chunk_to_embed", "doc_uri"],
  num_results=3)
docs = results.get('result', {}).get('data_array', [])
docs